# Run SDK-Backed Data Agent Evaluation

This notebook runs the official `fabric-data-agent-sdk` against a deployed Fabric Data Agent and captures raw baseline or final evidence. It does not simulate answers or calculate the hackathon's reviewed 24-point score.

Use `NB_Review_And_Score_Data_Agent.ipynb` after both snapshots to inspect source/query evidence and calculate the deterministic scorecard.

## Required Fabric setup

Before running the notebook:

1. In the configuration cell, set `DOMAIN_PROFILE` to the deployed domain, for example `water-utilities`.
2. In the notebook **Explorer** pane, select **Add data items** and attach the Lakehouse named by that domain's deployment.
3. Make that Lakehouse the notebook's **default Lakehouse**.
4. Set `SNAPSHOT_NAME` to `baseline` before tuning and `final` after tuning.
5. Set `TABLE_NAME` if needed. The SDK creates and appends to this Delta table; you do not create it manually.

With the default settings, results are written to a domain-specific evaluation table and a matching run-steps table. If the Lakehouse does not support schemas, both tables appear at the root of **Tables** instead.

In [ ]:
# 1. Install compatible SDK dependencies, then restart the Fabric Python interpreter.

import subprocess

import sys



packages = [

    "fabric-data-agent-sdk>=0.1.30a0",

    "pandas",

    "typing_extensions>=4.12.2",

    "PyJWT>=2.6.0",

]

subprocess.check_call(

    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]

)



verification = subprocess.run(

    [

        sys.executable,

        "-c",

        "from fabric.dataagent.evaluation import "

        "evaluate_data_agent, get_evaluation_details, get_evaluation_summary; "

        "from typing_extensions import Sentinel; "

        "print('Fabric Data Agent evaluation SDK dependencies verified')",

    ],

    capture_output=True,

    text=True,

)

if verification.returncode != 0:

    raise RuntimeError(

        "The packages installed, but a fresh Python process could not import them.\n"

        f"{verification.stderr.strip()}"

    )

print(verification.stdout.strip())

print("Restarting the Fabric Python interpreter; execution will continue in the next cell.")

notebookutils.session.restartPython()


## 2. Configure and verify the restarted kernel

In [ ]:
import importlib

# Domain parameters
DOMAIN_PROFILE = "water-utilities"
DOMAIN_SETTINGS = {
    "water-utilities": {"lakehouse": "WaterUtilitiesDemo", "agent": "WaterUtilitiesOperationsAgent"},
}
if DOMAIN_PROFILE not in DOMAIN_SETTINGS:
    raise ValueError(
        f"No notebook defaults exist for {DOMAIN_PROFILE!r}. Add its Lakehouse and Data Agent names to DOMAIN_SETTINGS."
    )
DEFAULT_LAKEHOUSE_NAME = DOMAIN_SETTINGS[DOMAIN_PROFILE]["lakehouse"]
AGENT_NAME = DOMAIN_SETTINGS[DOMAIN_PROFILE]["agent"]
DOMAIN_TOKEN = DOMAIN_PROFILE.replace("-", "_")

fabric_evaluation = importlib.import_module("fabric.dataagent.evaluation")
fabric_runtime = importlib.import_module("fabric.dataagent._fabric_runtime")

from typing_extensions import Sentinel

print("Current notebook kernel import verified:", fabric_evaluation.__name__)
fabric_context = fabric_runtime.get_fabric_context()
default_lakehouse_id = fabric_context.get("trident.lakehouse.id")
default_lakehouse_filesystem = fabric_context.get("fs.defaultFS")
if not default_lakehouse_id or not default_lakehouse_filesystem:
    raise RuntimeError(
        "No default Lakehouse is attached. In the Explorer pane, select Add data items, "
        f"attach {DEFAULT_LAKEHOUSE_NAME}, set it as the default Lakehouse, and rerun from this cell."
    )
print("Domain profile:", DOMAIN_PROFILE)
print("Default Lakehouse verified:", default_lakehouse_id)

# Evaluator parameters
WORKSPACE_NAME = "Hackathon"
DATASET_NAME = "challenge"  # use the domain challenge or the Step 5 "routing" extension
SNAPSHOT_NAME = "final"  # use "baseline" before tuning and "final" after tuning
INCLUDE_PARAPHRASES = True  # doubles challenge prompts; routing remains unexpanded
# The SDK creates/appends this table in the attached default Lakehouse.
TABLE_NAME = f"{DOMAIN_TOKEN}_evaluation_{SNAPSHOT_NAME}"
DATA_AGENT_STAGE = "sandbox"  # "sandbox"/"draft" for unpublished agents; "production" after publish
CRITIC_PROMPT = ""  # optional stricter/domain-specific evaluator prompt
REPOSITORY_OWNER = "hSushmithaShetty13"
REPOSITORY_NAME = "Water-Utilities"
REPOSITORY_REF = "main"
OUTPUT_PATH = f"{DOMAIN_PROFILE}_{SNAPSHOT_NAME}_sdk_evaluation_results.json"
SAVE_OFFICIAL_DETAILS_CSV = True

## 3. Download the question dataset

Download the versioned challenge or routing dataset directly from this repository. No repository Python module is required.

In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd
import requests

raw_base_url = (
    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"
    f"{REPOSITORY_NAME}/{REPOSITORY_REF}"
)
dataset_url = f"{raw_base_url}/evaluation/{DATASET_NAME}/{DOMAIN_PROFILE}.json"

dataset_response = requests.get(dataset_url, timeout=180)
dataset_response.raise_for_status()
dataset = dataset_response.json()

if INCLUDE_PARAPHRASES and DATASET_NAME == "challenge":
    expanded_queries = []
    for item in dataset["evaluation_queries"]:
        original = {**item, "original_id": item["id"], "variant": "original"}
        paraphrase = {
            **item,
            "id": f"{item['id']}-P",
            "original_id": item["id"],
            "question": item["paraphrase"],
            "variant": "paraphrase",
        }
        expanded_queries.extend([original, paraphrase])
    dataset["evaluation_queries"] = expanded_queries
    dataset["metadata"]["total_queries"] = len(expanded_queries)

dataset_path = Path(tempfile.gettempdir()) / (
    f"{SNAPSHOT_NAME}_{DATASET_NAME}_{DOMAIN_PROFILE}.json"
)
dataset_path.write_text(json.dumps(dataset, indent=2), encoding="utf-8")

print("Dataset:", dataset_url)
print("Snapshot:", SNAPSHOT_NAME)
print("SDK prompts:", len(dataset["evaluation_queries"]))
print("Dataset saved to:", dataset_path)

## 4. Run the real SDK-backed snapshot



Run once with `SNAPSHOT_NAME = "baseline"` before tuning and again with `SNAPSHOT_NAME = "final"` after tuning. With `INCLUDE_PARAPHRASES = True`, the SDK evaluates all 12 challenge prompts.

In [ ]:
valid_data_agent_stages = {"sandbox", "draft", "production"}
data_agent_stage = DATA_AGENT_STAGE.strip().lower()
if data_agent_stage not in valid_data_agent_stages:
    raise ValueError(
        f"DATA_AGENT_STAGE must be one of {sorted(valid_data_agent_stages)}; "
        f"received {DATA_AGENT_STAGE!r}."
    )

sdk_input_df = pd.DataFrame(
    {
        "question": [item["question"] for item in dataset["evaluation_queries"]],
        "expected_answer": [item["ground_truth_answer"] for item in dataset["evaluation_queries"]],
    }
)

evaluation_kwargs = {
    "workspace_name": WORKSPACE_NAME or None,
    "table_name": TABLE_NAME,
    "data_agent_stage": data_agent_stage,
}
if CRITIC_PROMPT:
    evaluation_kwargs["critic_prompt"] = CRITIC_PROMPT

evaluation_id = fabric_evaluation.evaluate_data_agent(
    sdk_input_df,
    AGENT_NAME,
    **evaluation_kwargs,
)
if evaluation_id is None:
    raise RuntimeError(
        "The Fabric SDK did not return an evaluation ID. Check the agent name, workspace, "
        "stage, and default Lakehouse, then rerun this cell."
    )

sdk_summary_df = fabric_evaluation.get_evaluation_summary(
    table_name=TABLE_NAME,
    verbose=False,
)
sdk_details_df = fabric_evaluation.get_evaluation_details(
    evaluation_id=evaluation_id,
    table_name=TABLE_NAME,
    get_all_rows=True,
    verbose=False,
)
if sdk_details_df is None:
    raise RuntimeError(
        f"The evaluation ran, but no detail rows were returned. Confirm that {DEFAULT_LAKEHOUSE_NAME} "
        "is attached as the default Lakehouse and rerun from section 2."
    )

print("Evaluation ID:", evaluation_id)
display(sdk_summary_df)
display(sdk_details_df)

## 5. Save the snapshot and evidence

The JSON records snapshot metadata plus the raw official summary and detail rows. The CSV preserves every official detail column for facilitator review.

In [ ]:
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

snapshot_payload = {
    "snapshot_name": SNAPSHOT_NAME,
    "dataset_name": DATASET_NAME,
    "agent_name": AGENT_NAME,
    "workspace_name": WORKSPACE_NAME or None,
    "data_agent_stage": data_agent_stage,
    "evaluation_id": str(evaluation_id),
    "table_name": TABLE_NAME,
    "question_count": len(dataset["evaluation_queries"]),
    "official_summary": json.loads(sdk_summary_df.to_json(orient="records")),
    "official_details": json.loads(sdk_details_df.to_json(orient="records")),
}
output_path.write_text(json.dumps(snapshot_payload, indent=2), encoding="utf-8")
print("Snapshot JSON:", output_path.resolve())

if SAVE_OFFICIAL_DETAILS_CSV:
    csv_path = output_path.with_name(f"{output_path.stem}_official_details.csv")
    sdk_details_df.to_csv(csv_path, index=False)
    print("Official detail CSV:", csv_path.resolve())